# This Notebook tests the effect of mask IoU on the analysis results. 

Methodology:
- Run calibration sequence using best model with variable shrinking factor: 0.4 --> 1.2
- Store std and slope
- Plot the curves and check for which shrinking factor is optimal

In [ ]:
chip_region = 1
EXPERIMENT_CONFIG = {
    "data_dir": "../../data/Raw/Chip2-NaCl 0-19_",           # str, Image directory
    "image_batch": f"290-4-0.75x-{chip_region}",             # str, Specific image batch, e.g. 290-4-0.75x-3
    "zoom": 0.75,                                            # float, Camera zoom
    "cc": True,                                              # bool, Use camera calibration
    "ffc": True,                                             # bool, Use flatfield correction
    "clahe": False,                                          # bool, Use clahe (for segmentation only)
    "model": "SAM2",                                  # str, SAM2 or CellposeSAM
    "approx": True,                                          # bool, Use approximated masks
    "approx_method": "ellipse",                              # str, "circle" or "ellipse"
    "shrinking_factor": {                                    # dict[str, int], scaling factor applied to approximated masks
        "large": 0.9,
        "medium": 0.8, 
        "small": 0.6, 
    },
    "overlay_masks": False,                                  # bool, use masks from first image on all following images
    "individual_intensities": False,                         # bool, show individual intensity points on intensity plots and link to segmentation results
    "filter_config": {                                       # dict, contains various configuration parameters for the segmentation mask filter
        "circularity_threshold": 0.7,                        # int, the circularity score used to filter irregular masks
        "border_thres": 2,                                   # int, the min number of pixels a mask has to be away from the border
        "max_groups": 3 if chip_region == 1 or chip_region == 3 else 2,                                     # int (2 or 3), the number of well sizes (2 if chip region 
        "eps_area": 40, 
        "eps_coords_medium": 90,
        "eps_coords_small": 70,
        "eps_coords_small_exclude": 20,
        "rough_only": False,
        "debug": False
    },
    "calibrator_config": {
        "ref_radii": {
            "region1": 500, # use large wells
            "region2": 190, # use medium wells
            "region3": 500 # use large wells
        },
        "schematic_path1": "../../data/Raw/Chip-schematic/chip_cleaned_1_10.jpg",
        "schematic_path2": "../../data/Raw/Chip-schematic/chip_cleaned_2_8.jpg",
        "schematic_path3": "../../data/Raw/Chip-schematic/chip_cleaned_3_10.jpg",
        "calculate_individual": False,
        "chip_region": chip_region,
        "debug": False
    },
    "debug": True
}

In [ ]:
from analysis.data_preprocessing import data_preprocessing
from analysis.camera_calibration import *
import json

camera = Camera(
    zoom=EXPERIMENT_CONFIG["zoom"],
    um_per_pixel=6.45
)

# Collect the right images from the specified directory and group them in batches of increasing NaCl concentration
data = data_preprocessing(EXPERIMENT_CONFIG["data_dir"], EXPERIMENT_CONFIG["image_batch"])

print(json.dumps(data, indent=4))

# Isolate the specified batch as a list of dictionaries
data = data[EXPERIMENT_CONFIG["image_batch"] + ".jpg"]

In [ ]:
from segmentation.segmenter import *

pretrained_checkpoint = "../../sam2/checkpoints/sam2.1_hiera_base_plus.pt"
finetuned_checkpoint = "../../models/TrainClean/IoULoss/checkpoints/checkpoint_230.pt"
model_cfg_b_plus = "configs/sam2.1/sam2.1_hiera_b+.yaml"

# Specify model checkpoint and config (finetuned checkpoints use the b+ config)
sam2 = SAM2Wrapper(config=model_cfg_b_plus, checkpoint=finetuned_checkpoint, points_per_side=128)
cellposeSAM = CellposeSAMWrapper()

if EXPERIMENT_CONFIG["model"] == "SAM2":
    model = sam2
elif EXPERIMENT_CONFIG["model"] == "CellposeSAM":
    model = cellposeSAM
else:
    raise ValueError("Invalid model passed in EXPERIMENT_CONFIG")

model = Segmenter(model=model, filter_config=EXPERIMENT_CONFIG["filter_config"])

In [ ]:
from analysis.noise_correction import FlatFieldCorrector
from analysis.signal_enhancement import apply_clahe
import os
import cv2
from PIL import Image
import numpy as np
from tqdm.notebook import tqdm
from segmentation.segmenter import *

corrector = FlatFieldCorrector()

setup_image_path = data[0]["file_path"]
setup_image = np.array(Image.open(setup_image_path).convert('L'))

schematic_path1 = EXPERIMENT_CONFIG["calibrator_config"]["schematic_path1"] 
schematic_raw1 = np.array(Image.open(schematic_path1).convert('L'))
schematic_mask1 = extract_schematic(image=schematic_raw1, debug=EXPERIMENT_CONFIG["debug"])
schematic_path2 = EXPERIMENT_CONFIG["calibrator_config"]["schematic_path2"] 
schematic_raw2 = np.array(Image.open(schematic_path2).convert('L'))
schematic_mask2 = extract_schematic(image=schematic_raw2, debug=EXPERIMENT_CONFIG["debug"])
schematic_path3 = EXPERIMENT_CONFIG["calibrator_config"]["schematic_path3"] 
schematic_raw3 = np.array(Image.open(schematic_path3).convert('L'))
schematic_mask3 = extract_schematic(image=schematic_raw3, debug=EXPERIMENT_CONFIG["debug"])
ref_images = {
    "region1": schematic_mask1,
    "region2": schematic_mask2,
    "region3": schematic_mask3,
}

calibrator = CameraCalibrator(
    setup_image=setup_image,
    camera=camera,
    segmenter=model,
    ref_images=ref_images,
    ref_radii=EXPERIMENT_CONFIG["calibrator_config"]["ref_radii"],
    calculate_individual=EXPERIMENT_CONFIG["calibrator_config"]["calculate_individual"]
)

preprocessed_data = []

def apply_preprocessing(image, cc: bool = True, ffc: bool = True, clahe: bool = True):
    result = image

    if cc and ffc:
        corrected, _, _ = corrector.apply_flatfield_correction(image)
        calibrated, _, _ = calibrator.undistort(corrected, chip_region=EXPERIMENT_CONFIG["calibrator_config"]["chip_region"], debug=EXPERIMENT_CONFIG["calibrator_config"]["debug"])
        result = calibrated
    else:
        if cc:
            calibrated, _, _ = calibrator.undistort(image, chip_region=EXPERIMENT_CONFIG["calibrator_config"]["chip_region"], debug=EXPERIMENT_CONFIG["calibrator_config"]["debug"])
            result = calibrated
        elif ffc:
            corrected, flatfield, darkfield = corrector.apply_flatfield_correction(image)
            result = corrected

    # Apply clahe 
    if clahe:
        result = apply_clahe(result)
    
    # Visualize Results
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(image, cmap='gray')
    plt.title('Original Image')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(result, cmap='gray')
    plt.title(f'Preprocessed Image: cc: {cc}, ffc: {ffc}, clahe: {clahe}')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    return result


for image_data in tqdm(data):
    # Open Image
    gray_image = np.array(Image.open(image_data["file_path"]).convert("L"))

    # Noise Correction and Signal Enhancement
    preprocessed = apply_preprocessing(gray_image, EXPERIMENT_CONFIG["cc"], EXPERIMENT_CONFIG["ffc"], EXPERIMENT_CONFIG["clahe"])

    # Store Results for the next step
    preprocessed_data.append({
        "image_original": gray_image,
        "image_enhanced": preprocessed,
        "file_path": image_data["file_path"],
        "nacl_percentage": image_data["nacl_percentage"]
    })



In [ ]:
from segmentation.mask_container import MaskContainer
segmented = False

segmentation_data = []
for item in tqdm(preprocessed_data):
    image_enhanced = item["image_enhanced"]
    print(f"==== NaCl concentration: {item['nacl_percentage']} ====")

    if EXPERIMENT_CONFIG["overlay_masks"]:
        if not segmented:
            model.generate_masks(image_enhanced)
            model.visualize_masks()
            model.filter_masks()
            model.visualize_masks_by_well_size()
            large_mc, medium_mc, small_mc = model.large_masks, model.medium_masks, model.small_masks
            segmented = True
        model.set_image(image_enhanced)
        model.large_masks, model.medium_masks, model.small_masks = large_mc, medium_mc, small_mc
    else:
        model.generate_masks(image_enhanced)
        model.visualize_masks()
        model.filter_masks()

    model.visualize_masks_by_well_size()
    
    segmentation_data.append({
        "image": item["image_original"],
        "image_enhanced": image_enhanced,
        "nacl_percentage": item["nacl_percentage"],
        "masks_large": model.large_masks,
        "masks_medium": model.medium_masks,
        "masks_small": model.small_masks
    })

In [ ]:
for item in tqdm(segmentation_data):
    item["masks_large"].use_approximated(EXPERIMENT_CONFIG["approx"], EXPERIMENT_CONFIG["approx_method"], shrink_factor=EXPERIMENT_CONFIG["shrinking_factor"]["large"])
    item["masks_medium"].use_approximated(EXPERIMENT_CONFIG["approx"], EXPERIMENT_CONFIG["approx_method"], shrink_factor=EXPERIMENT_CONFIG["shrinking_factor"]["medium"])
    item["masks_small"].use_approximated(EXPERIMENT_CONFIG["approx"], EXPERIMENT_CONFIG["approx_method"], shrink_factor=EXPERIMENT_CONFIG["shrinking_factor"]["small"])
            

postprocessed_data = segmentation_data    

In [ ]:
from analysis.analysis import intensity_analysis


intensity_data = []

for item in tqdm(postprocessed_data):
    intensity_data_large = intensity_analysis(item["image_enhanced"], item["masks_large"])
    intensity_data_medium = intensity_analysis(item["image_enhanced"], item["masks_medium"])
    intensity_data_small = intensity_analysis(item["image_enhanced"], item["masks_small"])

    intensity_data.append({
        "image": item["image"],
        "image_enhanced": item["image_enhanced"],
        "nacl_percentage": item["nacl_percentage"],
        "large": intensity_data_large,
        "medium": intensity_data_medium,
        "small": intensity_data_small
    })

In [ ]:
from analysis import utils
import pandas as pd

calibration_data = []
calibration_containers = {
    "large": [],
    "medium": [],
    "small": []
}
for item in tqdm(intensity_data):
    row = {
        'NaCl_percentage': item["nacl_percentage"],
        'image': item["image_enhanced"],
        'mean_intensity_large': item["large"]["mean_intensity"],
        'std_intensity_large': item["large"]["std_intensity"],
        'mean_intensity_medium': item["medium"]["mean_intensity"],
        'std_intensity_medium': item["medium"]["std_intensity"],
        'mean_intensity_small': item["small"]["mean_intensity"],
        'std_intensity_small': item["small"]["std_intensity"],
        'mean_intensity_all': (item["large"]["mean_intensity"] + item["medium"]["mean_intensity"] + item["small"]["mean_intensity"]) / 3,
        'std_intensity_all': (item["large"]["std_intensity"] + item["medium"]["std_intensity"] + item["small"]["std_intensity"]) / 3,
        'masks_large': item["large"]["masks"],
        'masks_medium': item["medium"]["masks"],
        'masks_small': item["small"]["masks"],
    }
    calibration_containers["large"].append(item["large"]["masks"])
    calibration_containers["medium"].append(item["medium"]["masks"])
    calibration_containers["small"].append(item["small"]["masks"])
    calibration_data.append(row)

df_data = pd.DataFrame(calibration_data).sort_values('NaCl_percentage')
# Fill NaN values with 0 if the entire column is NaN
df_data.loc[:, df_data.isna().all()] = 0 
# Compute refractive index (RI) using a simplified linear model
df_data['RI'] = 1.3330 + 0.0018 * df_data['NaCl_percentage']
# Drop rows with NaN values
df_calibration = df_data.dropna()

### Mask Visualization ###
utils.create_mask_plots(calibration_data, identify=EXPERIMENT_CONFIG["individual_intensities"])

avg_std_all = df_calibration["std_intensity_all"].astype(float).mean()
avg_std_large = df_calibration["std_intensity_large"].astype(float).mean()
avg_std_medium = df_calibration["std_intensity_medium"].astype(float).mean()
avg_std_small = df_calibration["std_intensity_small"].astype(float).mean()

print(f"==== Average std all: {avg_std_all} ==== ")
print(f"==== Average std large: {avg_std_large} ==== ")
print(f"==== Average std medium: {avg_std_medium} ==== ")
print(f"==== Average std small: {avg_std_small} ==== ")

settings = ""
if EXPERIMENT_CONFIG["ffc"]:
    settings += "ffc"
    
if EXPERIMENT_CONFIG["cc"]:
    settings += "cc"

if EXPERIMENT_CONFIG["approx"]:
    settings += "approx"
    
if settings == "":
    settings = "raw"

### Intensity vs RI Plots ###
slopes = utils.create_intensity_plots(
    df_calibration, 
    well_sizes=["large", "medium", "small"], 
    mask_containers=calibration_containers,
    individual_intensities=EXPERIMENT_CONFIG["individual_intensities"],
    settings=settings,
    csv_path="./summary.csv",
    model=EXPERIMENT_CONFIG["model"]
)

In [ ]:
print(slopes)

In [ ]:
import pandas as pd
import os

# File path
csv_file = f'cc_{chip_region}.csv'

# Check if file exists and load it, or create new DataFrame
if os.path.exists(csv_file):
    # Load existing CSV
    df = pd.read_csv(csv_file)
    print("Loaded existing CSV file")
else:
    # Create new DataFrame with column headers
    df = pd.DataFrame(columns=[
        'shrinking_factor', 
        'std_large', 
        'slope_large',
        'std_medium', 
        'slope_medium',
        'std_small', 
        'slope_small',
    ])
    print("Created new CSV file")

avg_slope_large = slopes["large"]
avg_slope_medium = slopes["medium"]
avg_slope_small = slopes["small"]

# Append a new row
new_row = {
    'shrinking_factor': EXPERIMENT_CONFIG["shrinking_factor"]["large"], 
    'std_large': avg_std_large, 
    'slope_large': avg_slope_large,
    'std_medium': avg_std_medium, 
    'slope_medium': avg_slope_medium,
    'std_small': avg_std_small, 
    'slope_small': avg_slope_small
}
df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

# Save back to CSV
df.to_csv(csv_file, index=False)
print(f"Saved data to {csv_file}")
print("\nCurrent data:")
print(df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

import pandas as pd
import matplotlib.pyplot as plt
import os

def load_and_plot_csv(csv_files):
    """
    Load one or multiple CSV files to DataFrames and create plots for slope and std vs shrinking_factor
    for each size (small, medium, large). Multiple files will be plotted on the same graphs.
    
    Parameters:
    csv_files (str or list): Path to a single CSV file or list of CSV file paths
    
    Returns:
    dict or pd.DataFrame: Dictionary of DataFrames (if multiple files) or single DataFrame
    """
    
    # Handle single file or multiple files
    if isinstance(csv_files, str):
        csv_files = [csv_files]
    
    # Dictionary to store DataFrames
    dataframes = {}
    
    # Load all CSV files
    for csv_file in csv_files:
        # Check if file exists
        if not os.path.exists(csv_file):
            print(f"Warning: File '{csv_file}' not found. Skipping.")
            continue
        
        # Load CSV file
        try:
            df = pd.read_csv(csv_file)
            # Sort by shrinking_factor to ensure proper line plotting
            df = df.sort_values('shrinking_factor').reset_index(drop=True)
            dataframes[csv_file] = df
            print(f"Successfully loaded {csv_file} - Shape: {df.shape}")
        except Exception as e:
            print(f"Error loading {csv_file}: {e}")
            continue
    
    if not dataframes:
        print("No valid CSV files were loaded.")
        return None
    
    # Verify expected columns exist in at least one file
    expected_columns = [
        'shrinking_factor', 
        'std_large', 
        'slope_large',
        'std_medium', 
        'slope_medium',
        'std_small', 
        'slope_small'
    ]
    
    # Check columns for each DataFrame
    for csv_file, df in dataframes.items():
        missing_columns = [col for col in expected_columns if col not in df.columns]
        if missing_columns:
            print(f"Warning: {csv_file} missing columns: {missing_columns}")
    
    # Create plots
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'Slope and Standard Deviation vs Shrinking Factor ({len(dataframes)} datasets)', fontsize=16)
    
    sizes = ['small', 'medium', 'large']
    
    # Define colors and markers for multiple datasets
    colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan']
    markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']
    linestyles = ['-', '--', '-.', ':', '-', '--', '-.', ':', '-', '--']
    
    # Plot for each size and each dataset
    for i, size in enumerate(sizes):
        # Plot slope vs shrinking_factor (top row)
        for j, (csv_file, df) in enumerate(dataframes.items()):
            if f'slope_{size}' in df.columns:
                color = colors[j % len(colors)]
                marker = markers[j % len(markers)]
                linestyle = linestyles[j % len(linestyles)]
                
                # Get filename without path for legend
                file_label = os.path.basename(csv_file).replace('.csv', '')
                
                axes[0, i].scatter(df['shrinking_factor'], df[f'slope_{size}'], 
                                 color=color, alpha=0.7, s=50, marker=marker, label=file_label)
                axes[0, i].plot(df['shrinking_factor'], df[f'slope_{size}'], 
                               color=color, alpha=0.7, linestyle=linestyle, linewidth=2)
        
        axes[0, i].set_title(f'Slope vs Shrinking Factor ({size.capitalize()})')
        axes[0, i].set_xlabel('Shrinking Factor')
        axes[0, i].set_ylabel('Slope')
        axes[0, i].grid(True, alpha=0.3)
        axes[0, i].legend(loc='upper right', framealpha=0.8)
        
        # Plot std vs shrinking_factor (bottom row)
        for j, (csv_file, df) in enumerate(dataframes.items()):
            if f'std_{size}' in df.columns:
                color = colors[j % len(colors)]
                marker = markers[j % len(markers)]
                linestyle = linestyles[j % len(linestyles)]
                
                # Get filename without path for legend
                file_label = os.path.basename(csv_file).replace('.csv', '')
                
                axes[1, i].scatter(df['shrinking_factor'], df[f'std_{size}'], 
                                 color=color, alpha=0.7, s=50, marker=marker, label=file_label)
                axes[1, i].plot(df['shrinking_factor'], df[f'std_{size}'], 
                               color=color, alpha=0.7, linestyle=linestyle, linewidth=2)
        
        axes[1, i].set_title(f'Standard Deviation vs Shrinking Factor ({size.capitalize()})')
        axes[1, i].set_xlabel('Shrinking Factor')
        axes[1, i].set_ylabel('Standard Deviation')
        axes[1, i].grid(True, alpha=0.3)
        axes[1, i].legend(loc='upper right', framealpha=0.8)
    
    plt.tight_layout()
    plt.show()
    
    # Return single DataFrame if only one file, otherwise return dictionary
    if len(dataframes) == 1:
        return list(dataframes.values())[0]
    else:
        return dataframes


# Load and plot the data
files = ['none_1.csv', 'none_3.csv', 'ffc_1.csv', 'ffc_3.csv', 'cc_ffc_3.csv', 'cc_ffc_1.csv', 'cc_1.csv', 'cc_3.csv']
dataframes = load_and_plot_csv(files)

In [ ]:
def load_and_plot_csv(csv_files, output_dir=None, save_individual_plots=True):
    """
    Load one or multiple CSV files to DataFrames and create plots for slope and std vs shrinking_factor
    for each size (small, medium, large). Multiple files will be plotted on the same graphs.
    Optionally saves individual plots as SVG files without legends.
    
    Parameters:
    csv_files (str or list): Path to a single CSV file or list of CSV file paths
    output_dir (str, optional): Directory to save individual SVG plots. If None, plots are not saved individually.
    save_individual_plots (bool): Whether to save individual plots as SVG files (default: True)
    
    Returns:
    dict or pd.DataFrame: Dictionary of DataFrames (if multiple files) or single DataFrame
    """
    import os
    import pandas as pd
    import matplotlib.pyplot as plt
    
    # Handle single file or multiple files
    if isinstance(csv_files, str):
        csv_files = [csv_files]
    
    # Dictionary to store DataFrames
    dataframes = {}
    
    # Load all CSV files
    for csv_file in csv_files:
        # Check if file exists
        if not os.path.exists(csv_file):
            print(f"Warning: File '{csv_file}' not found. Skipping.")
            continue
        
        # Load CSV file
        try:
            df = pd.read_csv(csv_file)
            # Sort by shrinking_factor to ensure proper line plotting
            df = df.sort_values('shrinking_factor').reset_index(drop=True)
            dataframes[csv_file] = df
            print(f"Successfully loaded {csv_file} - Shape: {df.shape}")
        except Exception as e:
            print(f"Error loading {csv_file}: {e}")
            continue
    
    if not dataframes:
        print("No valid CSV files were loaded.")
        return None
    
    # Verify expected columns exist in at least one file
    expected_columns = [
        'shrinking_factor', 
        'std_large', 
        'slope_large',
        'std_medium', 
        'slope_medium',
        'std_small', 
        'slope_small'
    ]
    
    # Check columns for each DataFrame
    for csv_file, df in dataframes.items():
        missing_columns = [col for col in expected_columns if col not in df.columns]
        if missing_columns:
            print(f"Warning: {csv_file} missing columns: {missing_columns}")
    
    # Create output directory if specified and doesn't exist
    if output_dir and save_individual_plots:
        os.makedirs(output_dir, exist_ok=True)
        print(f"Individual plots will be saved to: {output_dir}")
    
    # Create combined plots
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'Slope and Standard Deviation vs Shrinking Factor ({len(dataframes)} datasets)', fontsize=16)
    
    sizes = ['small', 'medium', 'large']
    
    # Define colors and markers for multiple datasets
    colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan']
    markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']
    linestyles = ['-', '--', '-.', ':', '-', '--', '-.', ':', '-', '--']
    
    # Plot for each size and each dataset
    for i, size in enumerate(sizes):
        # Plot slope vs shrinking_factor (top row)
        for j, (csv_file, df) in enumerate(dataframes.items()):
            if f'slope_{size}' in df.columns:
                color = colors[j % len(colors)]
                marker = markers[j % len(markers)]
                linestyle = linestyles[j % len(linestyles)]
                
                # Get filename without path for legend
                file_label = os.path.basename(csv_file).replace('.csv', '')
                
                axes[0, i].scatter(df['shrinking_factor'], df[f'slope_{size}'], 
                                 color=color, alpha=0.7, s=50, marker=marker, label=file_label)
                axes[0, i].plot(df['shrinking_factor'], df[f'slope_{size}'], 
                               color=color, alpha=0.7, linestyle=linestyle, linewidth=2)
        
        axes[0, i].set_title(f'Slope vs Shrinking Factor ({size.capitalize()})')
        axes[0, i].set_xlabel('Shrinking Factor')
        axes[0, i].set_ylabel('Slope')
        axes[0, i].grid(True, alpha=0.3)
        axes[0, i].legend(loc='upper right', framealpha=0.8)
        
        # Plot std vs shrinking_factor (bottom row)
        for j, (csv_file, df) in enumerate(dataframes.items()):
            if f'std_{size}' in df.columns:
                color = colors[j % len(colors)]
                marker = markers[j % len(markers)]
                linestyle = linestyles[j % len(linestyles)]
                
                # Get filename without path for legend
                file_label = os.path.basename(csv_file).replace('.csv', '')
                
                axes[1, i].scatter(df['shrinking_factor'], df[f'std_{size}'], 
                                 color=color, alpha=0.7, s=50, marker=marker, label=file_label)
                axes[1, i].plot(df['shrinking_factor'], df[f'std_{size}'], 
                               color=color, alpha=0.7, linestyle=linestyle, linewidth=2)
        
        axes[1, i].set_title(f'Standard Deviation vs Shrinking Factor ({size.capitalize()})')
        axes[1, i].set_xlabel('Shrinking Factor')
        axes[1, i].set_ylabel('Standard Deviation')
        axes[1, i].grid(True, alpha=0.3)
        axes[1, i].legend(loc='upper right', framealpha=0.8)
    
    plt.tight_layout()
    plt.show()
    
    # Save individual plots as SVG files (without legends)
    if output_dir and save_individual_plots:
        print("\nSaving individual SVG plots...")
        
        # Create individual plots for each metric and size combination
        plot_configs = [
            ('slope', 'Slope vs Shrinking Factor'),
            ('std', 'Standard Deviation vs Shrinking Factor')
        ]
        
        for metric, ylabel_base in plot_configs:
            for size in sizes:
                # Create individual figure
                fig_individual, ax = plt.subplots(1, 1, figsize=(8, 6))
                
                # Plot data for all datasets
                for j, (csv_file, df) in enumerate(dataframes.items()):
                    column_name = f'{metric}_{size}'
                    if column_name in df.columns:
                        color = colors[j % len(colors)]
                        marker = markers[j % len(markers)]
                        linestyle = linestyles[j % len(linestyles)]
                        
                        ax.scatter(df['shrinking_factor'], df[column_name], 
                                 color=color, alpha=0.7, s=50, marker=marker)
                        ax.plot(df['shrinking_factor'], df[column_name], 
                               color=color, alpha=0.7, linestyle=linestyle, linewidth=2)
                
                # Set labels and title
                ax.set_title(f'{ylabel_base} ({size.capitalize()})')
                ax.set_xlabel('Shrinking Factor')
                ax.set_ylabel(ylabel_base.split(' vs ')[0])  # Extract metric name
                ax.grid(True, alpha=0.3)
                
                # No legend for individual plots
                
                # Save as SVG
                filename = f'{metric}_{size}_vs_shrinking_factor.svg'
                filepath = os.path.join(output_dir, filename)
                
                plt.tight_layout()
                plt.savefig(filepath, format='svg', bbox_inches='tight', dpi=300)
                plt.close(fig_individual)  # Close to free memory
                
                print(f"Saved: {filepath}")
        
        print(f"\nAll individual plots saved to: {output_dir}")
    
    # Return single DataFrame if only one file, otherwise return dictionary
    if len(dataframes) == 1:
        return list(dataframes.values())[0]
    else:
        return dataframes

output_dir = './plots'
df = load_and_plot_csv(
    files,
    output_dir=output_dir,
    save_individual_plots=True
)